# Embeddings and Similarity

**Embeddings** are dense numerical vectors that represent the meaning of text. Two sentences with similar meanings will have vectors that are close together in vector space — regardless of the specific words used.

In this notebook we:
1. Create sentence embeddings with `sentence-transformers`
2. Compute cosine similarity between sentence pairs
3. Visualize a similarity heatmap
4. Build a simple nearest-neighbour search

## Setup

In [ ]:
!pip install transformers datasets sentence-transformers pillow -q

## Imports

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import matplotlib.pyplot as plt
import torch

print("Libraries loaded.")

## 1. Creating Sentence Embeddings

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "AI is transforming the world.",
    "Machine learning is changing everything.",
    "The cat sat on the mat.",
    "A kitten rested on a rug.",
    "Deep learning uses neural networks.",
    "I love pizza and pasta.",
]

embeddings = model.encode(sentences)
print(f"Encoded {len(sentences)} sentences")
print(f"Embedding shape: {embeddings.shape}  (sentences × dimensions)")

## 2. Cosine Similarity

**Cosine similarity** measures the angle between two vectors, ranging from -1 (opposite) to 1 (identical). For normalized embeddings it equals the dot product.

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Compare specific pairs
pairs = [
    (0, 1),  # AI / ML — semantically close
    (2, 3),  # cat / kitten — semantically close
    (0, 5),  # AI / pizza — semantically distant
    (2, 5),  # cat / pizza — unrelated
]

print("Sentence pair similarities:\n")
for i, j in pairs:
    sim = cosine_similarity(embeddings[i], embeddings[j])
    print(f"  {sim:.3f}  '{sentences[i]}'")
    print(f"         '{sentences[j]}'\n")

## 3. Similarity Heatmap

Visualize the full pairwise similarity matrix to see which sentences cluster together.

In [ ]:
n = len(sentences)
sim_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = cosine_similarity(embeddings[i], embeddings[j])

# Short labels for display
labels = [s[:30] + "..." if len(s) > 30 else s for s in sentences]

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(sim_matrix, cmap="RdYlGn", vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label="Cosine Similarity")
ax.set_xticks(range(n)); ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(n)); ax.set_yticklabels(labels, fontsize=8)
ax.set_title("Sentence Similarity Heatmap")

for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{sim_matrix[i,j]:.2f}", ha="center", va="center", fontsize=7)

plt.tight_layout()
plt.show()

## 4. Find the Most Similar Sentence

Given a query, rank all sentences by similarity — the core idea behind semantic search.

In [ ]:
def find_most_similar(query, corpus_sentences, corpus_embeddings, top_k=3):
    query_emb = model.encode([query])[0]
    scores = [(cosine_similarity(query_emb, emb), sent)
              for emb, sent in zip(corpus_embeddings, corpus_sentences)]
    scores.sort(reverse=True)
    return scores[:top_k]

queries = [
    "How do neural networks learn?",
    "I enjoy Italian food.",
    "Cats are adorable animals.",
]

for query in queries:
    print(f"Query: '{query}'")
    results = find_most_similar(query, sentences, embeddings)
    for rank, (score, sent) in enumerate(results, 1):
        print(f"  {rank}. [{score:.3f}] {sent}")
    print()